# Auto-Classification Demo

This notebook demonstrates the full implementation of Snowflake's native **Auto-Classification** feature to meet enterprise data governance tagging requirements.

**What this demo covers:**
1. Create governance database and custom tags
2. Grant required privileges
3. Create a custom classifier (regex-based NZ IRD number detection)
4. Create tag-map table linking semantic categories to custom tags
5. Create classification profiles with tag maps and custom classifiers
6. Create sample test data with sensitive information
7. Attach profile and run classification
8. Verify tag assignments and monitor tag coverage
9. Tag propagation (table → schema → database)
10. Tag-based masking policies
11. Cleanup

> **Prerequisites:** ACCOUNTADMIN role, Enterprise Edition or higher, `SNOWFLAKE.DATA_PRIVACY` schema available.

---
## Step 1: Create Governance Database and Tags

We create a dedicated `GOVERNANCE` database to store all custom tags, classification profiles, and mapping tables.

In [ ]:
# Run this once per kernel session to enable %%sql magic
%load_ext sql

# Fix ipython-sql + prettytable style compatibility
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [ ]:
import os

# Uses ~/.snowflake/connections.toml
# Example profile name in connections.toml: [auto_classification_demo]
connection_name = os.getenv('SNOWFLAKE_CONNECTION_NAME', 'default')

# Connect ipython-sql via Snowflake named connection
%sql snowflake:///?connection_name=$connection_name

print(f"Connected using Snowflake connection profile: {connection_name}")

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

-- Create the governance database
CREATE DATABASE IF NOT EXISTS GOVERNANCE
  COMMENT = 'Central governance database for tags, classification profiles, and monitoring';

In [ ]:
%%sql
USE DATABASE GOVERNANCE;
USE SCHEMA PUBLIC;

### Create Column-Level Tags

These tags are applied automatically by the classification engine based on the semantic categories detected in column data.

In [ ]:
%%sql
-- Column-level tags (applied by auto-classification)
CREATE TAG IF NOT EXISTS GOVERNANCE.PUBLIC.PERSONAL_INFO
  ALLOWED_VALUES 'True'
  COMMENT = 'Indicates column contains personal information';

CREATE TAG IF NOT EXISTS GOVERNANCE.PUBLIC.PCI_DATA
  ALLOWED_VALUES 'True'
  COMMENT = 'Indicates column contains PCI (Payment Card Industry) data';

CREATE TAG IF NOT EXISTS GOVERNANCE.PUBLIC.DATA_SENSITIVITY
  ALLOWED_VALUES 'Sensitive'
  COMMENT = 'Indicates column contains sensitive data';

-- Custom classification tag (applied when custom classifiers detect matches)
CREATE TAG IF NOT EXISTS GOVERNANCE.PUBLIC.CUSTOM_CLASSIFIED
  ALLOWED_VALUES 'NZ_IRD_NUMBER'
  COMMENT = 'Indicates column matched a custom classifier rule (e.g., NZ IRD numbers)';

### Create Multi-Level Tags (Table, Schema, Database)

These tags are applied at table/view, schema, and database levels — either manually from EDM metadata or via the propagation procedure.

In [ ]:
%%sql
-- Multi-level tags (table/view, schema, database)
CREATE TAG IF NOT EXISTS GOVERNANCE.PUBLIC.INFORMATION_CLASSIFICATION
  ALLOWED_VALUES 'Highly restricted', 'Restricted', 'Standard', 'Public'
  COMMENT = 'Information classification level per enterprise data governance policy';

CREATE TAG IF NOT EXISTS GOVERNANCE.PUBLIC.DATA_SUBJECT
  COMMENT = 'Data subject(s) as defined in the Enterprise Data Model';

---
## Step 2: Grant Required Privileges

Auto-classification requires specific privileges. We create a functional role for managing classification.

In [ ]:
%%sql
-- Create a functional role for classification management
CREATE ROLE IF NOT EXISTS DATA_CLASSIFIER;

-- Grant privileges for tag management
GRANT CREATE TAG ON SCHEMA GOVERNANCE.PUBLIC TO ROLE DATA_CLASSIFIER;
GRANT APPLY TAG ON ACCOUNT TO ROLE DATA_CLASSIFIER;

-- Grant classification execution privileges
GRANT DATABASE ROLE SNOWFLAKE.CLASSIFICATION_ADMIN TO ROLE DATA_CLASSIFIER;

-- Grant the role to ACCOUNTADMIN for this demo
GRANT ROLE DATA_CLASSIFIER TO ROLE ACCOUNTADMIN;

---
## Step 3: Create Custom Classifier

Custom classifiers extend Snowflake's built-in classification with organization-specific patterns. Here we create a custom classifier for **NZ IRD Numbers** (Inland Revenue Department tax identifiers) using a regex pattern.

This demonstrates how to detect domain-specific sensitive data that built-in classifiers may not cover.

In [ ]:
%%sql
-- Create a custom classifier for NZ IRD Numbers
-- IRD numbers follow the pattern: XX-XXX-XXX (8 digits with hyphens)
CREATE OR REPLACE SNOWFLAKE.DATA_PRIVACY.CUSTOM_CLASSIFIER
  GOVERNANCE.PUBLIC.NZ_IRD_CLASSIFIER();

In [ ]:
%%sql
-- Add regex pattern for NZ IRD Numbers to the custom classifier
CALL GOVERNANCE.PUBLIC.NZ_IRD_CLASSIFIER!ADD_REGEX(
  SEMANTIC_CATEGORY => 'NZ_IRD_NUMBER',
  PRIVACY_CATEGORY => 'IDENTIFIER',
  VALUE_REGEX => '\\d{2}-\\d{3}-\\d{3}',
  COL_NAME_REGEX => '.*IRD.*',
  DESCRIPTION => 'Detects NZ IRD numbers in XX-XXX-XXX format',
  THRESHOLD => 0.6
);

---
## Step 4: Create Tag-Map Table

This mapping table defines how Snowflake's semantic categories map to our custom tags. The classification engine uses this to apply the correct tags automatically.

In [ ]:
%%sql
-- Create the tag map table
CREATE OR REPLACE TABLE GOVERNANCE.PUBLIC.TAG_MAP (
  TAG_DATABASE     STRING,
  TAG_SCHEMA       STRING,
  TAG_NAME         STRING,
  TAG_VALUE        STRING,
  SEMANTIC_CATEGORY STRING,
  PRIVACY_CATEGORY STRING
);

-- Insert mappings: PERSONAL_INFO tag
INSERT INTO GOVERNANCE.PUBLIC.TAG_MAP VALUES
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','NAME','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','EMAIL','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','PHONE_NUMBER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','STREET_ADDRESS','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','PASSPORT','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','DRIVERS_LICENSE','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','NATIONAL_IDENTIFIER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','BANK_ACCOUNT','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','TAX_IDENTIFIER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','DATE_OF_BIRTH','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','AGE','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','GENDER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','ETHNICITY','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','MARITAL_STATUS','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','IP_ADDRESS','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','IMEI','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','SALARY','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','NZ_IRD_NUMBER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','NAME','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','EMAIL','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','PHONE_NUMBER','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','STREET_ADDRESS','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','POSTAL_CODE','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','CITY','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','LATITUDE','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','LONGITUDE','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','COUNTRY','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','OCCUPATION','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','URL','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','VIN','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','ADMINISTRATIVE_AREA_1','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PERSONAL_INFO','True','ORGANIZATION_IDENTIFIER','QUASI_IDENTIFIER');

-- Insert mappings: PCI_DATA tag
INSERT INTO GOVERNANCE.PUBLIC.TAG_MAP VALUES
  ('GOVERNANCE','PUBLIC','PCI_DATA','True','PAYMENT_CARD','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PCI_DATA','True','BANK_ACCOUNT','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PCI_DATA','True','PAYMENT_CARD','QUASI_IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','PCI_DATA','True','BANK_ACCOUNT','QUASI_IDENTIFIER');

-- Insert mappings: DATA_SENSITIVITY tag
INSERT INTO GOVERNANCE.PUBLIC.TAG_MAP VALUES
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','NAME','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','EMAIL','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','PHONE_NUMBER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','STREET_ADDRESS','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','PASSPORT','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','DRIVERS_LICENSE','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','NATIONAL_IDENTIFIER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','BANK_ACCOUNT','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','TAX_IDENTIFIER','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','PAYMENT_CARD','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','SALARY','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','IP_ADDRESS','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','IMEI','IDENTIFIER'),
  ('GOVERNANCE','PUBLIC','DATA_SENSITIVITY','Sensitive','NZ_IRD_NUMBER','IDENTIFIER');

-- Insert mappings: CUSTOM_CLASSIFIED tag (from custom classifier)
INSERT INTO GOVERNANCE.PUBLIC.TAG_MAP VALUES
  ('GOVERNANCE','PUBLIC','CUSTOM_CLASSIFIED','NZ_IRD_NUMBER','NZ_IRD_NUMBER','IDENTIFIER');

In [ ]:
%%sql
-- Verify the tag map
SELECT TAG_NAME, COUNT(*) AS mapping_count
FROM GOVERNANCE.PUBLIC.TAG_MAP
GROUP BY TAG_NAME
ORDER BY TAG_NAME;

---
## Step 5: Create Classification Profiles

Classification Profiles tell Snowflake's auto-classification engine *which* semantic categories to look for and *what tags* to apply when found.

We create the main combined profile that covers PII, PCI, NZ-specific categories, **and custom classifiers**.

In [ ]:
%%sql
-- Detach profile from database if previously attached (needed for re-runs)
-- Only run this if CLASSIFICATION_DEMO database already exists with a profile attached
BEGIN
  LET db_exists INT := (SELECT COUNT(*) FROM INFORMATION_SCHEMA.DATABASES WHERE DATABASE_NAME = 'CLASSIFICATION_DEMO');
  IF (:db_exists > 0) THEN
    ALTER DATABASE CLASSIFICATION_DEMO UNSET CLASSIFICATION_PROFILE;
  END IF;
EXCEPTION
  WHEN OTHER THEN
    -- Ignore errors (profile may not be set)
    NULL;
END;

In [ ]:
%%sql
-- Create the main classification profile (PII + PCI + NZ combined)
CREATE OR REPLACE SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE
  GOVERNANCE.PUBLIC.AUTO_CLASSIFICATION_PROFILE(
    {
      'minimum_object_age_for_classification_days': 0,
      'maximum_classification_validity_days': 30,
      'auto_tag': true,
      'classify_views': false,
      'tag_map': {
        'column_tag_map': [
          {
            'tag_name': 'GOVERNANCE.PUBLIC.PERSONAL_INFO',
            'tag_value': 'True',
            'semantic_categories': ['NAME','EMAIL','PHONE_NUMBER','STREET_ADDRESS','PASSPORT','DRIVERS_LICENSE','NATIONAL_IDENTIFIER','NZ_IRD_NUMBER','BANK_ACCOUNT','TAX_IDENTIFIER','DATE_OF_BIRTH','AGE','GENDER','ETHNICITY','MARITAL_STATUS','IP_ADDRESS','IMEI','SALARY','OCCUPATION','COUNTRY','POSTAL_CODE','CITY','LATITUDE','LONGITUDE','URL','VIN','ADMINISTRATIVE_AREA_1','ORGANIZATION_IDENTIFIER']
          },
          {
            'tag_name': 'GOVERNANCE.PUBLIC.PCI_DATA',
            'tag_value': 'True',
            'semantic_categories': ['PAYMENT_CARD','BANK_ACCOUNT']
          },
          {
            'tag_name': 'GOVERNANCE.PUBLIC.DATA_SENSITIVITY',
            'tag_value': 'Sensitive',
            'semantic_categories': ['NAME','EMAIL','PHONE_NUMBER','STREET_ADDRESS','PASSPORT','DRIVERS_LICENSE','NATIONAL_IDENTIFIER','NZ_IRD_NUMBER','BANK_ACCOUNT','TAX_IDENTIFIER','PAYMENT_CARD','SALARY','IP_ADDRESS','IMEI']
          }
        ]
      },
      'custom_classifiers': {
        'nz_ird_classifier': GOVERNANCE.PUBLIC.NZ_IRD_CLASSIFIER!LIST()
      }
    }
  );

---
## Step 6: Create Test Data

We create a sample database with tables containing NZ-specific sensitive data to demonstrate classification detection.

In [ ]:
%%sql
-- Create a test database for classification demo
CREATE DATABASE IF NOT EXISTS CLASSIFICATION_DEMO
  COMMENT = 'Test database for auto-classification demo';

CREATE SCHEMA IF NOT EXISTS CLASSIFICATION_DEMO.CUSTOMERS;

In [ ]:
%%sql
-- Create a customer table with PII data (NZ-specific)
CREATE OR REPLACE TABLE CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS (
  CUSTOMER_ID       NUMBER AUTOINCREMENT PRIMARY KEY,
  FULL_NAME         VARCHAR(200),
  EMAIL_ADDRESS     VARCHAR(200),
  PHONE             VARCHAR(50),
  STREET_ADDRESS    VARCHAR(300),
  CITY              VARCHAR(100),
  REGION            VARCHAR(100),
  POSTCODE          VARCHAR(10),
  DATE_OF_BIRTH     DATE,
  IRD_NUMBER        VARCHAR(20),
  PASSPORT_NUMBER   VARCHAR(20),
  DRIVERS_LICENSE   VARCHAR(20),
  GENDER            VARCHAR(20),
  ETHNICITY         VARCHAR(50),
  OCCUPATION        VARCHAR(100)
);

-- Insert sample NZ data
INSERT INTO CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS
  (FULL_NAME, EMAIL_ADDRESS, PHONE, STREET_ADDRESS, CITY, REGION, POSTCODE,
   DATE_OF_BIRTH, IRD_NUMBER, PASSPORT_NUMBER, DRIVERS_LICENSE, GENDER, ETHNICITY, OCCUPATION)
VALUES
  ('Aroha Williams','aroha.williams@email.co.nz','+64 21 456 7890','45 Lambton Quay','Wellington','Wellington Region','6011','1985-03-15','12-345-678','LA123456','WN123456','Female','Maori','Software Engineer'),
  ('James Henderson','james.h@gmail.com','+64 27 890 1234','12 Queen Street','Auckland','Auckland Region','1010','1990-07-22','98-765-432','LB654321','AK654321','Male','NZ European','Accountant'),
  ('Mei Lin Chen','mei.chen@company.co.nz','+64 22 111 2222','88 Colombo Street','Christchurch','Canterbury','8011','1978-11-08','45-678-901','LC789012','CH789012','Female','Asian','Doctor'),
  ('Rawiri Kopu','rawiri.k@xtra.co.nz','+64 29 333 4444','5 Devon Street','New Plymouth','Taranaki','4310','1995-01-30','67-890-123','LD345678','NP345678','Male','Maori','Teacher'),
  ('Sarah O''Brien','sarah.ob@outlook.co.nz','+64 21 555 6666','23 George Street','Dunedin','Otago','9016','1982-09-12','23-456-789','LE901234','DN901234','Female','NZ European','Nurse');

In [ ]:
%%sql
-- Create a payments table with PCI data
CREATE OR REPLACE TABLE CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS (
  PAYMENT_ID       NUMBER AUTOINCREMENT PRIMARY KEY,
  CUSTOMER_ID      NUMBER,
  CARD_NUMBER      VARCHAR(20),
  CARD_EXPIRY      VARCHAR(10),
  BANK_ACCOUNT_NUM VARCHAR(25),
  ACCOUNT_NAME     VARCHAR(200),
  ANNUAL_SALARY    NUMBER(12,2)
);

-- Insert sample PCI data
INSERT INTO CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS
  (CUSTOMER_ID, CARD_NUMBER, CARD_EXPIRY, BANK_ACCOUNT_NUM, ACCOUNT_NAME, ANNUAL_SALARY)
VALUES
  (1, '4111 1111 1111 1111', '12/26', '01-0123-0123456-00', 'Aroha Williams', 125000.00),
  (2, '5500 0000 0000 0004', '03/27', '02-0456-0456789-01', 'James Henderson', 95000.00),
  (3, '3400 0000 0000 009',  '08/25', '03-0789-0789012-02', 'Mei Lin Chen', 185000.00),
  (4, '4222 2222 2222 2222', '11/26', '06-0321-0654321-00', 'Rawiri Kopu', 72000.00),
  (5, '5100 0000 0000 0008', '06/27', '01-0987-0987654-03', 'Sarah O''Brien', 88000.00);

In [ ]:
%%sql
-- Create a non-sensitive table for comparison
CREATE OR REPLACE TABLE CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG (
  PRODUCT_ID    NUMBER AUTOINCREMENT PRIMARY KEY,
  PRODUCT_NAME  VARCHAR(200),
  CATEGORY      VARCHAR(100),
  PRICE         NUMBER(10,2),
  STOCK_QTY     NUMBER,
  DESCRIPTION   VARCHAR(1000)
);

INSERT INTO CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG
  (PRODUCT_NAME, CATEGORY, PRICE, STOCK_QTY, DESCRIPTION)
VALUES
  ('KiwiSaver Fund A', 'Investment', 0.00, 0, 'Conservative growth fund'),
  ('Home Loan Fixed 2yr', 'Lending', 0.00, 0, 'Fixed rate home loan product'),
  ('Everyday Checking', 'Banking', 5.00, 0, 'Standard everyday transaction account'),
  ('Credit Card Gold', 'Cards', 80.00, 0, 'Premium credit card with rewards'),
  ('Travel Insurance', 'Insurance', 45.00, 0, 'Comprehensive travel insurance policy');

---
## Step 7: Attach Classification Profile to Database

This is the key step that enables auto-classification. Once the profile is attached, Snowflake will automatically classify all existing and new tables in the database.

In [ ]:
%%sql
-- Attach the classification profile to the demo database
ALTER DATABASE CLASSIFICATION_DEMO
  SET CLASSIFICATION_PROFILE = 'GOVERNANCE.PUBLIC.AUTO_CLASSIFICATION_PROFILE';

> **Note:** Auto-classification runs asynchronously. The initial scan may take a few minutes depending on table sizes. You can also trigger classification manually (next step).

---
## Step 8: Run Manual Classification (Optional)

You can manually trigger classification to see immediate results without waiting for the async scan. This uses `SYSTEM$CLASSIFY` to classify individual tables.

In [ ]:
%%sql
-- Manually classify the customer table to see immediate results
CALL SYSTEM$CLASSIFY('CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS',
  'GOVERNANCE.PUBLIC.AUTO_CLASSIFICATION_PROFILE'
);

In [ ]:
%%sql
-- Manually classify the payments table
CALL SYSTEM$CLASSIFY('CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS',
  'GOVERNANCE.PUBLIC.AUTO_CLASSIFICATION_PROFILE'
);

In [ ]:
%%sql
-- Manually classify the non-sensitive product table
CALL SYSTEM$CLASSIFY('CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG',
  'GOVERNANCE.PUBLIC.AUTO_CLASSIFICATION_PROFILE'
);

---
## Step 9: Verify Tag Assignments

After classification, we verify the tags have been applied correctly to our columns.

In [ ]:
%%sql
-- Check all tags applied to the customer table
SELECT *
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS', 'TABLE'
  )
)
ORDER BY COLUMN_NAME, TAG_NAME;

In [ ]:
%%sql
-- Check tags on the payment table (should have PCI_DATA tags)
SELECT *
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS', 'TABLE'
  )
)
ORDER BY COLUMN_NAME, TAG_NAME;

In [ ]:
%%sql
-- Check tags on non-sensitive product table (should have minimal/no sensitive tags)
SELECT *
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG', 'TABLE'
  )
)
ORDER BY COLUMN_NAME, TAG_NAME;

---
## Step 10: Monitoring & Coverage Dashboard

These queries provide a governance overview of tag coverage across your classified objects.

In [ ]:
%%sql
-- Summary: Which columns have our custom governance tags?
-- Using INFORMATION_SCHEMA for real-time results (ACCOUNT_USAGE has ~2hr latency)
SELECT 
  TAG_DATABASE,
  TAG_SCHEMA,
  OBJECT_NAME AS TABLE_NAME,
  COLUMN_NAME,
  TAG_NAME,
  TAG_VALUE
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS', 'TABLE'
  )
)
WHERE TAG_NAME IN ('PERSONAL_INFO', 'PCI_DATA', 'DATA_SENSITIVITY')

UNION ALL

SELECT 
  TAG_DATABASE,
  TAG_SCHEMA,
  OBJECT_NAME AS TABLE_NAME,
  COLUMN_NAME,
  TAG_NAME,
  TAG_VALUE
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS', 'TABLE'
  )
)
WHERE TAG_NAME IN ('PERSONAL_INFO', 'PCI_DATA', 'DATA_SENSITIVITY')

ORDER BY TABLE_NAME, COLUMN_NAME, TAG_NAME;

In [ ]:
%%sql
-- Coverage metrics: percentage of columns tagged per table
WITH all_columns AS (
  SELECT 
    TABLE_CATALOG AS DB,
    TABLE_SCHEMA,
    TABLE_NAME,
    COUNT(*) AS total_columns
  FROM CLASSIFICATION_DEMO.INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_SCHEMA = 'CUSTOMERS'
  GROUP BY ALL
),
customer_tags AS (
  SELECT COUNT(DISTINCT COLUMN_NAME) AS cnt
  FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS','TABLE'))
  WHERE TAG_NAME IN ('PERSONAL_INFO','PCI_DATA','DATA_SENSITIVITY')
),
payment_tags AS (
  SELECT COUNT(DISTINCT COLUMN_NAME) AS cnt
  FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS','TABLE'))
  WHERE TAG_NAME IN ('PERSONAL_INFO','PCI_DATA','DATA_SENSITIVITY')
),
product_tags AS (
  SELECT COUNT(DISTINCT COLUMN_NAME) AS cnt
  FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG','TABLE'))
  WHERE TAG_NAME IN ('PERSONAL_INFO','PCI_DATA','DATA_SENSITIVITY')
)
SELECT 
  a.TABLE_NAME,
  a.total_columns,
  CASE a.TABLE_NAME
    WHEN 'CUSTOMER_DETAILS' THEN (SELECT cnt FROM customer_tags)
    WHEN 'PAYMENT_METHODS' THEN (SELECT cnt FROM payment_tags)
    WHEN 'PRODUCT_CATALOG' THEN (SELECT cnt FROM product_tags)
  END AS tagged_columns,
  ROUND(
    CASE a.TABLE_NAME
      WHEN 'CUSTOMER_DETAILS' THEN (SELECT cnt FROM customer_tags)
      WHEN 'PAYMENT_METHODS' THEN (SELECT cnt FROM payment_tags)
      WHEN 'PRODUCT_CATALOG' THEN (SELECT cnt FROM product_tags)
    END / a.total_columns * 100, 1
  ) AS coverage_pct
FROM all_columns a
ORDER BY a.TABLE_NAME;

---
## Step 11: Tag Propagation (Table-Level Information Classification)

We apply the **Information Classification Decision Matrix** to propagate column-level classification results up to the table level.

| If ANY column contains... | Table Classification |
|---|---|
| NAME, BANK_ACCOUNT, NATIONAL_IDENTIFIER, PASSPORT, DRIVERS_LICENSE, TAX_IDENTIFIER, PAYMENT_CARD, SALARY | **Highly Restricted** |
| EMAIL, PHONE_NUMBER, STREET_ADDRESS, IP_ADDRESS, IMEI, ORGANIZATION_IDENTIFIER, DATE_OF_BIRTH, AGE, GENDER, ETHNICITY, MARITAL_STATUS | **Restricted** |
| OCCUPATION, COUNTRY, POSTAL_CODE, CITY, LATITUDE, LONGITUDE, URL, VIN | **Standard** |
| No sensitive data detected | **Public** |

In [ ]:
%%sql
-- Create the propagation procedure
-- Note: This uses ACCOUNT_USAGE (has ~2hr latency). For immediate demo, use the manual cells below.
CREATE OR REPLACE PROCEDURE GOVERNANCE.PUBLIC.PROPAGATE_INFORMATION_CLASSIFICATION(
  P_DATABASE STRING
)
RETURNS STRING
LANGUAGE SQL
AS
$$
DECLARE
  v_count INTEGER DEFAULT 0;
  v_table_name STRING;
  v_classification STRING;
  v_check INTEGER;
  c_tables CURSOR FOR
    SELECT DISTINCT OBJECT_DATABASE || '.' || OBJECT_SCHEMA || '.' || OBJECT_NAME AS full_table_name
    FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
    WHERE OBJECT_DATABASE = :P_DATABASE
      AND DOMAIN = 'COLUMN'
      AND TAG_DATABASE = 'GOVERNANCE'
      AND TAG_NAME IN ('PERSONAL_INFO', 'PCI_DATA', 'DATA_SENSITIVITY');
BEGIN
  FOR rec IN c_tables DO
    v_table_name := rec.full_table_name;
    v_classification := 'Public';
    
    -- Check for Highly restricted categories
    SELECT COUNT(*) INTO :v_check
    FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
    WHERE OBJECT_DATABASE || '.' || OBJECT_SCHEMA || '.' || OBJECT_NAME = :v_table_name
      AND TAG_NAME = 'SEMANTIC_CATEGORY'
      AND TAG_VALUE IN ('NAME','BANK_ACCOUNT','NATIONAL_IDENTIFIER','PASSPORT','DRIVERS_LICENSE','TAX_IDENTIFIER','PAYMENT_CARD','SALARY');
    
    IF (v_check > 0) THEN
      v_classification := 'Highly restricted';
    ELSE
      -- Check for Restricted categories
      SELECT COUNT(*) INTO :v_check
      FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
      WHERE OBJECT_DATABASE || '.' || OBJECT_SCHEMA || '.' || OBJECT_NAME = :v_table_name
        AND TAG_NAME = 'SEMANTIC_CATEGORY'
        AND TAG_VALUE IN ('EMAIL','PHONE_NUMBER','STREET_ADDRESS','IP_ADDRESS','IMEI','ORGANIZATION_IDENTIFIER','DATE_OF_BIRTH','AGE','GENDER','ETHNICITY','MARITAL_STATUS');
      
      IF (v_check > 0) THEN
        v_classification := 'Restricted';
      ELSE
        -- Check for Standard categories
        SELECT COUNT(*) INTO :v_check
        FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES
        WHERE OBJECT_DATABASE || '.' || OBJECT_SCHEMA || '.' || OBJECT_NAME = :v_table_name
          AND TAG_NAME = 'SEMANTIC_CATEGORY'
          AND TAG_VALUE IN ('OCCUPATION','COUNTRY','POSTAL_CODE','CITY','LATITUDE','LONGITUDE','URL','VIN');
        
        IF (v_check > 0) THEN
          v_classification := 'Standard';
        END IF;
      END IF;
    END IF;
    
    -- Apply INFORMATION_CLASSIFICATION tag to the table
    EXECUTE IMMEDIATE 'ALTER TABLE ' || :v_table_name || 
      ' SET TAG GOVERNANCE.PUBLIC.INFORMATION_CLASSIFICATION = ''' || :v_classification || '''';
    
    v_count := v_count + 1;
  END FOR;
  
  RETURN 'Propagated INFORMATION_CLASSIFICATION to ' || v_count::STRING || ' tables';
END;
$$;

In [ ]:
%%sql
-- Alternative: Direct manual assignment based on what we know about our test tables
-- (Use this if the procedure encounters issues with ACCOUNT_USAGE latency)

-- CUSTOMER_DETAILS has NAME, PASSPORT, DRIVERS_LICENSE, etc. → Highly restricted
ALTER TABLE CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS
  SET TAG GOVERNANCE.PUBLIC.INFORMATION_CLASSIFICATION = 'Highly restricted';

-- PAYMENT_METHODS has PAYMENT_CARD, BANK_ACCOUNT, SALARY → Highly restricted
ALTER TABLE CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS
  SET TAG GOVERNANCE.PUBLIC.INFORMATION_CLASSIFICATION = 'Highly restricted';

-- PRODUCT_CATALOG has no sensitive data → Public
ALTER TABLE CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG
  SET TAG GOVERNANCE.PUBLIC.INFORMATION_CLASSIFICATION = 'Public';

In [ ]:
%%sql
-- Apply DATA_SUBJECT tags at the table level
ALTER TABLE CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS
  SET TAG GOVERNANCE.PUBLIC.DATA_SUBJECT = 'Customer';

ALTER TABLE CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS
  SET TAG GOVERNANCE.PUBLIC.DATA_SUBJECT = 'Customer';

In [ ]:
%%sql
-- Verify table-level tags
SELECT 
  OBJECT_NAME,
  TAG_NAME,
  TAG_VALUE,
  DOMAIN
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES(
    'CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS', 'TABLE'
  )
)
UNION ALL
SELECT 
  OBJECT_NAME,
  TAG_NAME,
  TAG_VALUE,
  DOMAIN
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES(
    'CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS', 'TABLE'
  )
)
UNION ALL
SELECT 
  OBJECT_NAME,
  TAG_NAME,
  TAG_VALUE,
  DOMAIN
FROM TABLE(
  INFORMATION_SCHEMA.TAG_REFERENCES(
    'CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG', 'TABLE'
  )
)
ORDER BY OBJECT_NAME, TAG_NAME;

---
## Step 12: Cost Monitoring

Monitor auto-classification costs to stay within budget.

In [ ]:
%%sql
-- Check auto-classification costs
SELECT 
  USAGE_DATE,
  SERVICE_TYPE,
  CREDITS_USED,
  CREDITS_USED_COMPUTE,
  CREDITS_USED_CLOUD_SERVICES
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AUTO_CLASSIFICATION'
ORDER BY USAGE_DATE DESC
LIMIT 30;

---
## Step 13: Governance Report — Full Tag Inventory

A comprehensive report showing all tags applied across the demo database.

In [ ]:
%%sql
-- Full governance report: all custom tags on all objects in the demo database
-- Using INFORMATION_SCHEMA for real-time results
SELECT 
  'CUSTOMER_DETAILS' AS TABLE_NAME,
  COLUMN_NAME,
  TAG_DATABASE || '.' || TAG_SCHEMA || '.' || TAG_NAME AS FULL_TAG_NAME,
  TAG_VALUE,
  DOMAIN
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS','TABLE'))
WHERE TAG_DATABASE = 'GOVERNANCE'

UNION ALL

SELECT 
  'PAYMENT_METHODS' AS TABLE_NAME,
  COLUMN_NAME,
  TAG_DATABASE || '.' || TAG_SCHEMA || '.' || TAG_NAME AS FULL_TAG_NAME,
  TAG_VALUE,
  DOMAIN
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS','TABLE'))
WHERE TAG_DATABASE = 'GOVERNANCE'

UNION ALL

SELECT 
  'PRODUCT_CATALOG' AS TABLE_NAME,
  COLUMN_NAME,
  TAG_DATABASE || '.' || TAG_SCHEMA || '.' || TAG_NAME AS FULL_TAG_NAME,
  TAG_VALUE,
  DOMAIN
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('CLASSIFICATION_DEMO.CUSTOMERS.PRODUCT_CATALOG','TABLE'))
WHERE TAG_DATABASE = 'GOVERNANCE'

ORDER BY TABLE_NAME, DOMAIN DESC, COLUMN_NAME, FULL_TAG_NAME;

---
## Step 14: Tag-Based Masking Policy

We create a **masking policy** tied to the `DATA_SENSITIVITY` tag. Any column tagged with `DATA_SENSITIVITY = 'Sensitive'` will be automatically masked for all roles **except ACCOUNTADMIN**.

This is a tag-based masking policy — it attaches to the tag itself, not individual columns. When auto-classification tags a column as sensitive, the masking is enforced immediately without manual intervention.

In [ ]:
%%sql
-- First unset any existing masking policies from the tag (safe for re-runs)
BEGIN
  BEGIN
    ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY UNSET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_STRING_MASK;
  EXCEPTION WHEN OTHER THEN NULL;
  END;
  BEGIN
    ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY UNSET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_NUMBER_MASK;
  EXCEPTION WHEN OTHER THEN NULL;
  END;
  BEGIN
    ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY UNSET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_DATE_MASK;
  EXCEPTION WHEN OTHER THEN NULL;
  END;
END;

-- Create masking policy for string columns tagged as sensitive
CREATE OR REPLACE MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_STRING_MASK
  AS (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN val
    ELSE '********'
  END
  COMMENT = 'Masks string values for all roles except ACCOUNTADMIN';

In [ ]:
%%sql
-- Create a masking policy for numeric columns tagged as sensitive (e.g., SALARY)
CREATE OR REPLACE MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_NUMBER_MASK
  AS (val NUMBER) RETURNS NUMBER ->
  CASE
    WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN val
    ELSE -1
  END
  COMMENT = 'Masks numeric values for all roles except ACCOUNTADMIN';

In [ ]:
%%sql
-- Create a masking policy for date columns tagged as sensitive (e.g., DATE_OF_BIRTH)
CREATE OR REPLACE MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_DATE_MASK
  AS (val DATE) RETURNS DATE ->
  CASE
    WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN val
    ELSE '1900-01-01'::DATE
  END
  COMMENT = 'Masks date values for all roles except ACCOUNTADMIN';

In [ ]:
%%sql
-- Attach masking policies to the DATA_SENSITIVITY tag
-- One policy per data type: STRING, NUMBER, DATE
ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY
  SET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_STRING_MASK;

ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY
  SET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_NUMBER_MASK;

ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY
  SET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_DATE_MASK;

### Verify Masking Behavior

As ACCOUNTADMIN, we should see the full unmasked data. Then we test with a non-privileged role to confirm masking is enforced.

In [ ]:
%%sql
-- As ACCOUNTADMIN, we see full data (unmasked)
SELECT FULL_NAME, EMAIL_ADDRESS, PHONE, DRIVERS_LICENSE, DATE_OF_BIRTH
FROM CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS
LIMIT 5;

In [ ]:
%%sql
-- Also check numeric masking on salary
SELECT ACCOUNT_NAME, CARD_NUMBER, ANNUAL_SALARY
FROM CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS
LIMIT 5;

In [ ]:
%%sql
-- Create a test role to verify masking
CREATE ROLE IF NOT EXISTS DATA_ANALYST;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE DATA_ANALYST;
GRANT USAGE ON DATABASE CLASSIFICATION_DEMO TO ROLE DATA_ANALYST;
GRANT USAGE ON SCHEMA CLASSIFICATION_DEMO.CUSTOMERS TO ROLE DATA_ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA CLASSIFICATION_DEMO.CUSTOMERS TO ROLE DATA_ANALYST;
GRANT ROLE DATA_ANALYST TO ROLE ACCOUNTADMIN;

In [ ]:
%%sql
-- Switch to DATA_ANALYST role and query — sensitive columns should be masked
USE ROLE DATA_ANALYST;

SELECT FULL_NAME, EMAIL_ADDRESS, PHONE, DRIVERS_LICENSE, DATE_OF_BIRTH
FROM CLASSIFICATION_DEMO.CUSTOMERS.CUSTOMER_DETAILS
LIMIT 5;

In [ ]:
%%sql
-- Check numeric/PCI masking as DATA_ANALYST
SELECT ACCOUNT_NAME, CARD_NUMBER, ANNUAL_SALARY
FROM CLASSIFICATION_DEMO.CUSTOMERS.PAYMENT_METHODS
LIMIT 5;

In [ ]:
%%sql
-- Switch back to ACCOUNTADMIN for cleanup
USE ROLE ACCOUNTADMIN;

---
## Cleanup (Optional)

Run these cells only when you want to tear down the demo environment.

In [ ]:
%%sql
-- CLEANUP: Remove masking policies from tag, then unset classification profile
-- BEGIN
--   BEGIN
--     ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY UNSET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_STRING_MASK;
--   EXCEPTION WHEN OTHER THEN NULL;
--   END;
--   BEGIN
--     ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY UNSET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_NUMBER_MASK;
--   EXCEPTION WHEN OTHER THEN NULL;
--   END;
--   BEGIN
--     ALTER TAG GOVERNANCE.PUBLIC.DATA_SENSITIVITY UNSET MASKING POLICY GOVERNANCE.PUBLIC.SENSITIVE_DATE_MASK;
--   EXCEPTION WHEN OTHER THEN NULL;
--   END;
--   BEGIN
--     ALTER DATABASE CLASSIFICATION_DEMO UNSET CLASSIFICATION_PROFILE;
--   EXCEPTION WHEN OTHER THEN NULL;
--   END;
-- END;

In [ ]:
%%sql
-- CLEANUP: Drop demo objects
-- DROP DATABASE IF EXISTS CLASSIFICATION_DEMO;
-- DROP ROLE IF EXISTS DATA_ANALYST;

-- Optionally drop governance objects (uncomment if desired)
-- DROP MASKING POLICY IF EXISTS GOVERNANCE.PUBLIC.SENSITIVE_STRING_MASK;
-- DROP MASKING POLICY IF EXISTS GOVERNANCE.PUBLIC.SENSITIVE_NUMBER_MASK;
-- DROP MASKING POLICY IF EXISTS GOVERNANCE.PUBLIC.SENSITIVE_DATE_MASK;
-- DROP TAG IF EXISTS GOVERNANCE.PUBLIC.PERSONAL_INFO;
-- DROP TAG IF EXISTS GOVERNANCE.PUBLIC.PCI_DATA;
-- DROP TAG IF EXISTS GOVERNANCE.PUBLIC.DATA_SENSITIVITY;
-- DROP TAG IF EXISTS GOVERNANCE.PUBLIC.INFORMATION_CLASSIFICATION;
-- DROP TAG IF EXISTS GOVERNANCE.PUBLIC.DATA_SUBJECT;
-- DROP TABLE IF EXISTS GOVERNANCE.PUBLIC.TAG_MAP;
-- DROP SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE IF EXISTS GOVERNANCE.PUBLIC.AUTO_CLASSIFICATION_PROFILE;
-- DROP DATABASE IF EXISTS GOVERNANCE;

---
## Summary

This demo walked through the complete auto-classification implementation:

| Step | What We Did |
|------|-------------|
| 1 | Created `GOVERNANCE` database with 5 custom tags |
| 2 | Granted classification privileges via `DATA_CLASSIFIER` role |
| 3 | Built tag map linking semantic categories → custom tags |
| 4 | Created combined classification profile (`AUTO_CLASSIFICATION_PROFILE`) |
| 5 | Created test tables with NZ-specific PII and PCI data |
| 6 | Attached profile to database (enables async auto-classification) |
| 7 | Ran manual `SYSTEM$CLASSIFY` for immediate results |
| 8-9 | Verified tags on columns; queried coverage metrics |
| 10 | Propagated column classifications to table-level `INFORMATION_CLASSIFICATION` |
| 11 | Monitored classification costs |
| 12 | Generated full governance report |
| 13 | Created tag-based masking policies (STRING, NUMBER, DATE) tied to `DATA_SENSITIVITY` tag |
| 14 | Verified masking: ACCOUNTADMIN sees clear data, DATA_ANALYST sees masked values |

**Key Outcome:** Columns tagged `DATA_SENSITIVITY = 'Sensitive'` are automatically masked for all non-ACCOUNTADMIN roles — no per-column policy attachment needed.